# Streams

Welcome to the documentation the *Streams* part of *Kafi Streams*.

Streams is stream processing made natural.

Streams is based on *Kafka Streams* on the surface but makes stream processing 10x simpler.

## Your First Streams Topology

In Streams, you create a *processing topology* or just *topology* with an arbitrary number of *sources* (=inputs) and an arbitrary number of *sinks* (=outputs).

Typically, sources and sinks are Kafka topics (but in principle, any data source/sink).

So let's do something.

Our source is a Kafka topic of orders like this:
```json
{"key": 42,
 "value": {"id": 42,
           "product_id": 4711,
           "customer_id": 23,
           "ts": 1609459200000},
 "partition": 3,
 "offset": 47,
 "timestamp": [1, 1784196912929],
 "headers": null}
```

Our first humble aim is to group these orders by customer (`customer_id`).


## Setup

For simplicity, we use Kafi's emulated Kafka on your local disk.

In [11]:
import sys
sys.path.insert(1, "..")

import kafi.streams.streams
import importlib
importlib.reload(kafi.streams.streams)

from kafi.fs.local.local import Local
from kafi.streams.streams import Streams, start_streams_task

# "Connect" to local Kafka emulation
c = Local({"local": {"root.dir": "."}})

# Source + sink topic names
source_str = "orders"
sink_str = "orders_grouped"

# Create the Streams topology
tn = (
    Streams.source(c, source_str)
    .map(lambda r: r["value"])
    .group_by_agg(lambda r: r["customer_id"],
                  lambda r: r,
                  lambda agg_r, r: {"order_ids": agg_r["order_ids"] + [r["id"]],
                                    "product_ids": agg_r["product_ids"] + [r["product_id"]]},
                  {"order_ids": [], "product_ids": []},
                  lambda by, agg_r: {"customer_id": by,
                                     "order_ids": agg_r["order_ids"],
                                     "product_ids": agg_r["product_ids"]})
    .map(lambda r: {"key": r["customer_id"],
                    "value": r}).peek("sink")
    .sink(c, sink_str)
)


In [12]:
import random

class OrderGenerator:
    def __init__(self):
        self.order_id_int = 0
        self.customer_id_int = 0
        #
        self.ts_int = 0
        self.ts_step_int = 1

    def generate(self):
        message_dict = {
            "key": self.order_id_int,
            "value": {"id": self.order_id_int,
                      "product_id": random.randint(0, 100 - 1),
                      "customer_id": random.randint(0, 10 - 1),
                      "ts": self.ts_int},
        }
        #
        self.order_id_int += 1
        #
        self.ts_int += self.ts_step_int
        #
        return message_dict


In [13]:
# Build it
built_tn = Streams.build(tn)
#
c.retouch(source_str)
c.retouch(sink_str)
#
stop = start_streams_task(built_tn)

#

gen = OrderGenerator()
pr = c.producer(source_str)
#
for _ in range(10):
    message_dict = gen.generate()
    pr.produce(message_dict["value"], key=message_dict["key"])
#
pr.close()



'orders'

sink: {'key': 3, 'value': {'customer_id': 3, 'order_ids': [0], 'product_ids': [73]}}
sink: {'key': 1, 'value': {'customer_id': 1, 'order_ids': [4, 1], 'product_ids': [34, 11]}}
sink: {'key': 4, 'value': {'customer_id': 4, 'order_ids': [7, 2], 'product_ids': [20, 9]}}
sink: {'key': 2, 'value': {'customer_id': 2, 'order_ids': [5, 3], 'product_ids': [31, 44]}}
sink: {'key': 8, 'value': {'customer_id': 8, 'order_ids': [9, 6], 'product_ids': [49, 44]}}
sink: {'key': 6, 'value': {'customer_id': 6, 'order_ids': [8], 'product_ids': [68]}}


In [14]:
stop()

<coroutine object start_streams_task.<locals>._stop_fun at 0x1086f8450>

In [9]:
print(await Streams.cancel_all_tasks())


[<Task cancelling name='streams_thread_623b3eb5-3a53-4853-abb1-d268b2271723' coro=<streams() running at /Users/m0724822/github/kafi/docs/../kafi/streams/streams.py:94> wait_for=<Future cancelled>>, <Task cancelling name='streams_thread_00e8c2a9-263d-40e9-8257-85508a2ae78e' coro=<streams() running at /Users/m0724822/github/kafi/docs/../kafi/streams/streams.py:94> wait_for=<Future cancelled>>]


In [15]:
c.cat(sink_str)

[{'value': {'customer_id': 3, 'order_ids': [0], 'product_ids': [73]},
  'key': '3',
  'headers': None,
  'timestamp': (1, 1784624517118),
  'partition': 0,
  'offset': 0,
  'topic': 'orders_grouped'},
 {'value': {'customer_id': 1, 'order_ids': [4, 1], 'product_ids': [34, 11]},
  'key': '1',
  'headers': None,
  'timestamp': (1, 1784624517118),
  'partition': 0,
  'offset': 1,
  'topic': 'orders_grouped'},
 {'value': {'customer_id': 4, 'order_ids': [7, 2], 'product_ids': [20, 9]},
  'key': '4',
  'headers': None,
  'timestamp': (1, 1784624517118),
  'partition': 0,
  'offset': 2,
  'topic': 'orders_grouped'},
 {'value': {'customer_id': 2, 'order_ids': [5, 3], 'product_ids': [31, 44]},
  'key': '2',
  'headers': None,
  'timestamp': (1, 1784624517118),
  'partition': 0,
  'offset': 3,
  'topic': 'orders_grouped'},
 {'value': {'customer_id': 8, 'order_ids': [9, 6], 'product_ids': [49, 44]},
  'key': '8',
  'headers': None,
  'timestamp': (1, 1784624517118),
  'partition': 0,
  'offset': 4